Import libraries and functions

In [1]:
import numpy as np
import torch
import pyro
from torch import nn
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

from codebase_0 import (
    CANN, MLP, cann_training_loop, mlp_train_loop, DeviceLoader, set_seed
)

from gamma_bcann_hmc_nuts import (
    gamma_stratified_subsample, gamma_make_chain_inits, 
    gamma_run_nuts, gamma_time_nuts
)
from gamma_bcann_vi import (
    GammaBCANN, gamma_build_bcann_guide, 
    gamma_fit_glm_mle, gamma_bcann_svi_training_loop
)

from gamma_nuts_parallel import gamma_run_nuts_parallel

seed = 42
set_seed(seed)
pyro.clear_param_store()

/Users/thangmai/Downloads/UNIMELB/HONOURS RESEARCH (ACTL40001)/.venv-honours/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Load data and train test split

In [ ]:
device = torch.device("cpu")

torch.set_default_device("cpu")

dataset = pd.read_csv("freMTPL2sev_transformed.csv", index_col= 0) 
y = torch.tensor(dataset["ClaimAmount"].to_numpy(), dtype=torch.float32)
weights = torch.ones(len(y)) 
X = torch.tensor(
    dataset.drop(columns=["IDpol", "Exposure","ClaimAmount"]).to_numpy(), 
    dtype=torch.float32
)   # 48 columns

# Learning-test set split
X_learn, X_test, y_learn, y_test, w_learn, w_test = train_test_split(
    X, y, weights, test_size= 0.1, random_state= seed
)

# Training-CV set split
X_train, X_val, y_train, y_val, w_train, w_val = train_test_split(
    X_learn, y_learn, w_learn, test_size= 0.1, random_state= seed
)

learn_scaler = MinMaxScaler().fit(X_learn)
X_learn = learn_scaler.transform(X_learn)
X_test = learn_scaler.transform(X_test)

train_scaler = MinMaxScaler().fit(X_train)
X_train = train_scaler.transform(X_train)
X_val = train_scaler.transform(X_val)

def T(a):
    return torch.as_tensor(a, dtype=torch.float32, device=device)

X_train, X_val, X_learn, X_test = map(T, (X_train, X_val, X_learn, X_test))
y_train, y_val, y_learn, y_test = map(T, (y_train, y_val, y_learn, y_test))
w_train, w_val, w_learn, w_test = map(T, (w_train, w_val, w_learn, w_test))

torch.set_default_device(device)
train_loader = DeviceLoader(
        X_train, y_train, w_train,
        batch_size = 2056,
        shuffle= True
)
val_loader = DeviceLoader(
        X_val, y_val, w_val,
        batch_size= 10000
)
learn_loader = DeviceLoader(
        X_learn, y_learn, w_learn,
        batch_size= 10000
)

Initialise GammaBCANN, VI posterior (as guide) and create a chain object for NUTS

In [3]:
network_architecture = [20,15, 10]
model = GammaBCANN(
    in_features= 48,
    neural_net_params={"hidden_activation_nodes": network_architecture,
                        "hidden_activations": nn.Tanh}
).to(device)
glm_mle = gamma_fit_glm_mle(X_learn, y_learn, w_learn, random_state=seed)
guide = gamma_build_bcann_guide(model, X_learn[:256], y_learn[:256], glm_mle=glm_mle, rank=10)
bcann_history = gamma_bcann_svi_training_loop(
        model, guide, 
        learn_loader, 
        val_loader,
        n_epochs=90, 
        lr=5e-3, 
        val_every=100, 
        patience=10, 
        random_state=seed, 
        device = device)

inits = gamma_make_chain_inits(model, X_learn, y_learn, 
                            w_learn, num_chains=6, glm_mle=glm_mle,
                            guide=guide, how="svi_sample", seed=seed, device = device)

Shape warm-up: alpha held at 0.058 for 20 epochs
Epoch 1 | -ELBO/n: 10.0335 | val deviance: 1.4816 | alpha (median): 0.058 [warm-up]


NUTS test run to see how long it takes

In [13]:
gamma_time_nuts(model, X_learn, y_learn, w_learn, inits, device = device, n_iter = 50)

1.57 s / iteration on cuda:2  (early warmup iterations are not representative of tree depth)


1.5684886741638184

Run full NUTS

In [5]:
hmc_nuts_samples, hmc_nuts_info = gamma_run_nuts_parallel(
    dict(
        in_features=48, 
        neural_net_params={
            "hidden_activation_nodes": network_architecture,
            "hidden_activations": nn.Tanh
        }
    ),   # GammaBCANN(**model_kwargs)
    X_learn, y_learn, w_learn, 
    inits, 
    out_dir="gamma_nuts_chains",
    model_cls = GammaBCANN,
    num_warmup=1000, 
    num_samples=1000, 
    target_accept=0.8, 
    max_tree_depth=8,
    seed=seed, 
    device=device, 
    progress_every= 5,
    jit_compile = True
)

elapsed 0:41:13
chain 5: Sample: 100%|██████████| 2000/2000 [31:52,  1.05it/s, step size=5.76e-02, acc. prob=0.939]


Write to disk

In [ ]:
torch.save(
    {
        "samples": hmc_nuts_samples,
        "info": hmc_nuts_info,
        "config": dict(num_warmup=1000, num_samples=1000, target_accept=0.8,
                       max_tree_depth=8, seed=seed, n_chains=6),
    },
    "hmc_results/gamma_hmcnuts_full_set.pt",
)
